# Training notebook for MLP

This notebook implements training for MLP architectures for image classification.



In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Imports

In [2]:
import os
import random
import csv
import time

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

from tqdm import tqdm
import time

from PIL import Image, ImageFilter
import cv2


## Reproducibility

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)


Device: cuda


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
zip_path = "/content/drive/MyDrive/DLprojekt1/archive.zip"
!unzip -q "/content/drive/MyDrive/DLprojekt1/archive.zip" -d "/content/"

## Paths

In [6]:
# global variables and creating folders
DATA_DIR = "/content"

PROJECT_DIR = "/content/drive/MyDrive/DLprojekt1"

MODEL_DIR = os.path.join(PROJECT_DIR, "saved_models")
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, "checkpoints")
RESULT_DIR = os.path.join(PROJECT_DIR, "results")
PLOT_DIR = os.path.join(PROJECT_DIR, "plots")

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

RESULT_FILE = os.path.join(RESULT_DIR, "experiment_results.csv")

if not os.path.exists(RESULT_FILE):

    with open(RESULT_FILE,"w",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            "experiment",
            "lr",
            "optimizer",
            "dropout",
            "weight_decay",
            "augmentation",
            "test_accuracy"
        ])


## Data Augmentations

In [7]:

class SobelAugmentation:

    def __init__(self, alpha=0.5):
        self.alpha = alpha

    def __call__(self, img):

        img_np = np.array(img)

        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)

        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)

        sobel = np.sqrt(sobelx**2 + sobely**2)

        sobel = np.uint8(255 * sobel / np.max(sobel))

        sobel_rgb = np.stack([sobel]*3, axis=-1)

        blended = cv2.addWeighted(img_np, 1-self.alpha, sobel_rgb, self.alpha, 0)

        return Image.fromarray(blended)


class GaussianBlur(object):

    def __call__(self,img):

        return img.filter(ImageFilter.GaussianBlur(radius=1.0))


def get_transform(augmentation=None):

    transform_list = []

    if augmentation == "rotation":
        transform_list.append(transforms.RandomRotation(15))

    if augmentation == "color_jitter":
        transform_list.append(
            transforms.ColorJitter(
                brightness=0.2,
                contrast=0.2,
                saturation=0.2,
                hue=0.1
            )
        )

    if augmentation == "gaussian_blur":
        transform_list.append(GaussianBlur())

    if augmentation == "sobel":
        transform_list.append(SobelAugmentation())

    transform_list.extend([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4789, 0.4723, 0.4305),
            (0.2421, 0.2383, 0.2587)
        )
    ])

    return transforms.Compose(transform_list)


def mixup_data(x, y, alpha=0.2):

    lam = np.random.beta(alpha,alpha)
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(device)
    mixed_x = lam*x + (1-lam)*x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam

def mixup_loss(criterion, pred, y_a, y_b, lam):

    return lam*criterion(pred, y_a) + (1-lam)*criterion(pred, y_b)


## Data Loading

In [8]:
def load_data(augmentation=None, batch_size=256, subset_ratio=1.0):

    train_transform = get_transform(augmentation)

    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(
            (0.4789, 0.4723, 0.4305),
            (0.2421, 0.2383, 0.2587)
        )
    ])

    train_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "train"),
        transform=train_transform
    )

    valid_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "valid"),
        transform=test_transform
    )

    test_set = torchvision.datasets.ImageFolder(
        os.path.join(DATA_DIR, "test"),
        transform=test_transform
    )

    if subset_ratio < 1.0:

        subset_size = int(len(train_set)*subset_ratio)
        train_set = torch.utils.data.Subset(train_set, range(subset_size))

    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    valid_loader = DataLoader(valid_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, valid_loader, test_loader


## MLP Architecture

In [9]:
class MLP(nn.Module):

    def __init__(self, dropout):

        super().__init__()

        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(32*32*3, 384)
        self.bn1 = nn.BatchNorm1d(384)

        self.fc2 = nn.Linear(384, 192)
        self.bn2 = nn.BatchNorm1d(192)

        self.fc3 = nn.Linear(192, 10)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        x = self.flatten(x)

        x = F.gelu(self.bn1(self.fc1(x)))
        x = self.dropout(x)

        x = F.gelu(self.bn2(self.fc2(x)))
        x = self.dropout(x)

        x = self.fc3(x)

        return x

## Evaluation

In [10]:

def evaluate(model, loader):

    model.eval()
    device = next(model.parameters()).device

    total_loss = 0.0
    total_correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = F.cross_entropy(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_correct += (logits.argmax(1) == labels).sum().item()
            total += labels.size(0)

    return total_correct / total, total_loss / total

In [11]:
class EarlyStopping:

    def __init__(self,patience=5):
        self.patience = patience
        self.best_score = None
        self.counter = 0
        self.stop = False

    def step(self,score):

        if self.best_score is None:
            self.best_score = score

        elif score <= self.best_score:

            self.counter += 1

            if self.counter >= self.patience:
                self.stop = True

        else:

            self.best_score = score
            self.counter = 0

        return self.stop

## Training

In [12]:
def train_model(model, train_loader, val_loader, optimizer, epochs=30,
                use_mixup=False, name=None):

    criterion = nn.CrossEntropyLoss()
    early_stopping = EarlyStopping()

    history = {
        "train_loss":[],
        "val_loss":[],
        "val_acc":[],
    }

    best_acc = 0

    for epoch in range(epochs):

        start_time = time.time()

        model.train()
        running_loss = 0
        total = 0

        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}", unit="batch"):

            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            if use_mixup:

                images,targets_a,targets_b,lam = mixup_data(images,labels)
                outputs = model(images)
                loss = mixup_loss(criterion, outputs, targets_a, targets_b, lam)

            else:

                outputs = model(images)
                loss = criterion(outputs,labels)

            loss.backward()
            optimizer.step()
            running_loss += loss.item() * labels.size(0)
            total += labels.size(0)

        val_acc, val_loss = evaluate(model, val_loader)
        train_loss = running_loss / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        epoch_time = time.time() - start_time

        print("train_loss:", f"{train_loss:.6f}", ", val_loss:", f"{val_loss:.6f}",
              ", val_acc:", f"{val_acc:.6f}", ", time:", round(epoch_time, 2), "s\n")

        torch.save(
            model.state_dict(),
            os.path.join(CHECKPOINT_DIR,f"checkpoint_epoch_{name}_{epoch+1}.pt")
        )

        if val_acc > best_acc:
            best_acc = val_acc

        if early_stopping.step(val_acc):

          print("Early stopping triggered")
          break

    return history


## Run Experiment

In [13]:
def save_history_csv(history, experiment_name):

    filename = os.path.join(RESULT_DIR, f"{experiment_name}_history.csv")

    epochs = len(history['train_loss'])

    with open(filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['epoch', 'train_loss', 'val_loss', 'val_acc'])
        for i in range(epochs):
            writer.writerow([
                i+1,
                history['train_loss'][i],
                history['val_loss'][i],
                history['val_acc'][i]
            ])

    print(f"History saved in: {filename}")

In [14]:
def run_experiment(model_class, name, lr, optimizer_name, dropout, weight_decay, augmentation=None,
                   epochs=40, subset_ratio=1.0, use_mixup=False):

    print("Running:", name ,"\n")

    train_loader, val_loader, test_loader = load_data(
        augmentation=augmentation,
        subset_ratio=subset_ratio
    )

    model = model_class(dropout).to(device)

    if optimizer_name == "SGD":
        optimizer = optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)

    elif optimizer_name == "SGD_momentum":
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)

    elif optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = train_model(model, train_loader, val_loader, optimizer, epochs=epochs, use_mixup=use_mixup, name=name)

    test_acc, _ = evaluate(model, test_loader)
    save_history_csv(history, name)

    torch.save(
        model.state_dict(),
        os.path.join(MODEL_DIR,f"{name}.pt")
    )

    with open(RESULT_FILE,"a",newline="") as f:

        writer = csv.writer(f)
        writer.writerow([
            name,
            lr,
            optimizer_name,
            dropout,
            weight_decay,
            augmentation,
            test_acc
        ])

    print("Test accuracy:", test_acc)


## Experiments

Experiments with different training, regularization parameters and data augumentation methods

In [ ]:
run_experiment(
    model_class=MLP,
    name="mlp_lr_0.1",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_lr_0.01",
    lr=0.01,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_lr_0.001",
    lr=0.001,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_lr_0.1 



Epoch 1: 100%|██████████| 352/352 [00:33<00:00, 10.45batch/s]


train_loss: 1.819376 , val_loss: 1.683469 , val_acc: 0.387344 , time: 66.48 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00,  9.90batch/s]


train_loss: 1.683626 , val_loss: 1.621222 , val_acc: 0.410078 , time: 68.14 s



...

train_loss: 1.758423 , val_loss: 1.723309 , val_acc: 0.382544 , time: 65.91 s



Epoch 19: 100%|██████████| 352/352 [00:33<00:00, 10.66batch/s]


train_loss: 1.751805 , val_loss: 1.717891 , val_acc: 0.385644 , time: 64.81 s



Epoch 20: 100%|██████████| 352/352 [00:33<00:00, 10.61batch/s]


train_loss: 1.743150 , val_loss: 1.710400 , val_acc: 0.387456 , time: 66.58 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_lr_0.001_history.csv
Test accuracy: 0.3879888888888889


Best lr parameter: 0.1

In [ ]:
## to juz mamy -> lr_0.01
#run_experiment(
#    model_class=MLP,
#    name="optimizer_sgd",
#    lr=0.1,
#    optimizer_name="SGD",
#    dropout=0.3,
#    weight_decay=1e-4,
#    epochs=20
#)

run_experiment(
    model_class=MLP,
    name="mlp_optimizer_sgd_momentum",
    lr=0.1,
    optimizer_name="SGD_momentum",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

run_experiment(
    model_class=MLP,
    name="mlp_optimizer_adam",
    lr=0.1,
    optimizer_name="Adam",
    dropout=0.3,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_optimizer_sgd_momentum 



Epoch 1: 100%|██████████| 352/352 [00:35<00:00,  9.90batch/s]


train_loss: 1.831449 , val_loss: 1.680629 , val_acc: 0.390000 , time: 68.53 s



Epoch 2: 100%|██████████| 352/352 [00:34<00:00, 10.28batch/s]


train_loss: 1.671305 , val_loss: 1.609750 , val_acc: 0.416000 , time: 67.78 s



...

train_loss: 2.036224 , val_loss: 2.027980 , val_acc: 0.259422 , time: 67.33 s



Epoch 5: 100%|██████████| 352/352 [00:35<00:00,  9.83batch/s]


train_loss: 2.053793 , val_loss: 2.054615 , val_acc: 0.228778 , time: 68.53 s



Epoch 6: 100%|██████████| 352/352 [00:34<00:00, 10.28batch/s]


train_loss: 2.048678 , val_loss: 2.047219 , val_acc: 0.232978 , time: 67.82 s

Early stopping triggered
History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_optimizer_adam_history.csv
Test accuracy: 0.2345888888888889


Best optimizer parameter: SGD

In [15]:
run_experiment(
   model_class=MLP,
   name="mlp_dropout_01",
   lr=0.1,
   optimizer_name="SGD",
   dropout=0.1,
   weight_decay=1e-4,
   epochs=20
)

# run_experiment(
#     model_class=MLP,
#     name="mlp_dropout_03",
#     lr=0.1,
#     optimizer_name="SGD",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=20
# )

run_experiment(
    model_class=MLP,
    name="mlp_dropout_05",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.5,
    weight_decay=1e-4,
    epochs=20
)

Running: mlp_dropout_01 



Epoch 1: 100%|██████████| 352/352 [00:41<00:00,  8.44batch/s]


train_loss: 1.757247 , val_loss: 1.678242 , val_acc: 0.392722 , time: 76.77 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00, 10.06batch/s]


train_loss: 1.605597 , val_loss: 1.592610 , val_acc: 0.422600 , time: 69.61 s



...

train_loss: 1.496717 , val_loss: 1.501085 , val_acc: 0.453156 , time: 73.92 s



Epoch 19: 100%|██████████| 352/352 [00:36<00:00,  9.72batch/s]


train_loss: 1.481572 , val_loss: 1.512694 , val_acc: 0.451011 , time: 70.81 s



Epoch 20: 100%|██████████| 352/352 [00:37<00:00,  9.47batch/s]


train_loss: 1.474189 , val_loss: 1.516126 , val_acc: 0.450289 , time: 71.46 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_dropout_05_history.csv
Test accuracy: 0.44866666666666666


Best dropout parameter: 0.3


In [16]:
run_experiment(
   model_class=MLP,
   name="mlp_weight_decay_1e2",
   lr=0.1,
   optimizer_name="SGD",
   dropout=0.3,
   weight_decay=1e-2,
   epochs=20
)

# run_experiment(
#     model_class=MLP,
#     name="mlp_weight_decay_1e3",
#     lr=0.1,
#     optimizer_name="SGD",
#     dropout=0.3,
#     weight_decay=1e-4,
#     epochs=20
# )

run_experiment(
    model_class=MLP,
    name="mlp_dweight_decay_1e5",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-5,
    epochs=20
)

Running: mlp_weight_decay_1e2 



Epoch 1: 100%|██████████| 352/352 [00:36<00:00,  9.53batch/s]


train_loss: 1.821038 , val_loss: 1.712801 , val_acc: 0.381667 , time: 72.0 s



Epoch 2: 100%|██████████| 352/352 [00:38<00:00,  9.26batch/s]


train_loss: 1.716052 , val_loss: 1.693830 , val_acc: 0.389100 , time: 72.67 s



...

train_loss: 1.355361 , val_loss: 1.495052 , val_acc: 0.464244 , time: 69.76 s



Epoch 17: 100%|██████████| 352/352 [00:36<00:00,  9.56batch/s]


train_loss: 1.340796 , val_loss: 1.489860 , val_acc: 0.461900 , time: 70.91 s



Epoch 18: 100%|██████████| 352/352 [00:35<00:00,  9.81batch/s]


train_loss: 1.326502 , val_loss: 1.502817 , val_acc: 0.459744 , time: 71.11 s

Early stopping triggered
History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_dweight_decay_1e5_history.csv
Test accuracy: 0.45943333333333336


Best weight_decay parameter: 1e-4

DATA AUGUMENTATION EXPERIMENTS

In [17]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_color_jitter",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="color_jitter",
    epochs=20
)

Running: mlp_augmentation_color_jitter 



Epoch 1: 100%|██████████| 352/352 [01:21<00:00,  4.33batch/s]


train_loss: 1.856100 , val_loss: 1.696465 , val_acc: 0.381056 , time: 115.35 s



Epoch 2: 100%|██████████| 352/352 [01:13<00:00,  4.77batch/s]


train_loss: 1.726672 , val_loss: 1.651651 , val_acc: 0.402989 , time: 107.35 s



...

train_loss: 1.419708 , val_loss: 1.517277 , val_acc: 0.453644 , time: 108.05 s



Epoch 19: 100%|██████████| 352/352 [01:13<00:00,  4.79batch/s]


train_loss: 1.410143 , val_loss: 1.502742 , val_acc: 0.459056 , time: 108.36 s



Epoch 20: 100%|██████████| 352/352 [01:17<00:00,  4.55batch/s]


train_loss: 1.394569 , val_loss: 1.500948 , val_acc: 0.460622 , time: 112.09 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_color_jitter_history.csv
Test accuracy: 0.4605222222222222


In [18]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_blur",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="gaussian_blur",
    epochs=20
)

Running: mlp_augmentation_blur 



Epoch 1: 100%|██████████| 352/352 [00:41<00:00,  8.40batch/s]


train_loss: 1.822979 , val_loss: 1.687615 , val_acc: 0.387711 , time: 76.5 s



Epoch 2: 100%|██████████| 352/352 [00:41<00:00,  8.51batch/s]


train_loss: 1.698010 , val_loss: 1.651752 , val_acc: 0.402567 , time: 75.15 s



...

train_loss: 1.447777 , val_loss: 1.505307 , val_acc: 0.456978 , time: 75.54 s



Epoch 19: 100%|██████████| 352/352 [00:40<00:00,  8.67batch/s]


train_loss: 1.440516 , val_loss: 1.477704 , val_acc: 0.466256 , time: 74.23 s



Epoch 20: 100%|██████████| 352/352 [00:39<00:00,  8.95batch/s]


train_loss: 1.431261 , val_loss: 1.475823 , val_acc: 0.469756 , time: 74.19 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_blur_history.csv
Test accuracy: 0.46886666666666665


In [19]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_sobel",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="sobel",
    epochs=20
)

Running: mlp_augmentation_sobel 



Epoch 1: 100%|██████████| 352/352 [00:50<00:00,  6.92batch/s]


train_loss: 1.871054 , val_loss: 1.906995 , val_acc: 0.328789 , time: 86.77 s



Epoch 2: 100%|██████████| 352/352 [00:50<00:00,  6.99batch/s]


train_loss: 1.714356 , val_loss: 1.860082 , val_acc: 0.349022 , time: 84.34 s



...

train_loss: 1.443818 , val_loss: 1.774945 , val_acc: 0.375956 , time: 84.26 s



Epoch 10: 100%|██████████| 352/352 [00:53<00:00,  6.64batch/s]


train_loss: 1.416040 , val_loss: 1.815701 , val_acc: 0.367000 , time: 87.03 s



Epoch 11: 100%|██████████| 352/352 [00:50<00:00,  6.92batch/s]


train_loss: 1.398276 , val_loss: 1.842626 , val_acc: 0.364744 , time: 86.94 s

Early stopping triggered
History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_sobel_history.csv
Test accuracy: 0.3641888888888889


In [20]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_mixup",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    use_mixup=True,
    epochs=20
)

Running: mlp_augmentation_mixup 



Epoch 1: 100%|██████████| 352/352 [00:35<00:00,  9.99batch/s]


train_loss: 1.910065 , val_loss: 1.696381 , val_acc: 0.392289 , time: 71.45 s



Epoch 2: 100%|██████████| 352/352 [00:35<00:00, 10.05batch/s]


train_loss: 1.808303 , val_loss: 1.651887 , val_acc: 0.404100 , time: 68.99 s



...

train_loss: 1.562412 , val_loss: 1.508374 , val_acc: 0.459000 , time: 70.16 s



Epoch 19: 100%|██████████| 352/352 [00:37<00:00,  9.31batch/s]


train_loss: 1.550616 , val_loss: 1.493754 , val_acc: 0.461489 , time: 72.2 s



Epoch 20: 100%|██████████| 352/352 [00:35<00:00,  9.86batch/s]


train_loss: 1.549793 , val_loss: 1.488581 , val_acc: 0.465600 , time: 71.77 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_mixup_history.csv
Test accuracy: 0.46403333333333335


In [21]:
run_experiment(
    model_class=MLP,
    name="mlp_augmentation_rotation",
    lr=0.1,
    optimizer_name="SGD",
    dropout=0.3,
    weight_decay=1e-4,
    augmentation="rotation",
    epochs=20
)

Running: mlp_augmentation_rotation 



Epoch 1: 100%|██████████| 352/352 [00:43<00:00,  8.11batch/s]


train_loss: 1.846813 , val_loss: 1.712502 , val_acc: 0.378267 , time: 77.43 s



Epoch 2: 100%|██████████| 352/352 [00:43<00:00,  8.14batch/s]


train_loss: 1.723565 , val_loss: 1.639806 , val_acc: 0.403811 , time: 77.18 s



...

train_loss: 1.490014 , val_loss: 1.472069 , val_acc: 0.469556 , time: 77.08 s



Epoch 19: 100%|██████████| 352/352 [00:43<00:00,  8.17batch/s]


train_loss: 1.484773 , val_loss: 1.471751 , val_acc: 0.467378 , time: 77.98 s



Epoch 20: 100%|██████████| 352/352 [00:42<00:00,  8.27batch/s]


train_loss: 1.478549 , val_loss: 1.453181 , val_acc: 0.476200 , time: 78.01 s

History saved in: /content/drive/MyDrive/DLprojekt1/results/mlp_augmentation_rotation_history.csv
Test accuracy: 0.4732222222222222
